# Importing Required Libraries

In this section, the required Python libraries for data
processing, numerical computations, random sampling,
and visualization are imported into the environment.


In [1]:
# کتابخانه‌های اصلی پروژه

import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt

## 📥 Loading the Dataset



# Chunk Processing

Due to the extremely large size of the dataset,
the data is processed in chunks in order to reduce
memory consumption and improve performance.

In [3]:
# Process dataset in chunks

# Function for reading dataset in chunks

def read_in_chunks(file_path, chunk_size=1000000):

    for chunk in pd.read_csv(
        file_path,
        chunksize=chunk_size
    ):

        yield chunk

# Dataset Normalization

In this step, all text columns in the dataset are
normalized in order to unify Persian and Arabic
characters.

Because the dataset is extremely large, the file is
processed in chunks and a new normalized dataset is
generated for future operations.


In [11]:
# Function for normalizing Persian text

def normalize_text(text):

    if isinstance(text, str):

        # Replace Arabic characters with Persian characters
        text = text.replace('ي', 'ی')
        text = text.replace('ك', 'ک')

        # Remove extra spaces
        text = text.strip()

    return text



# Output file name

output_file = "normalized_bank_data.csv"


# To write header only once

first_chunk = True


# Process dataset in chunks

for chunk in read_in_chunks("Bank NIT DB.csv", 1000000):

    # Select text columns
    text_columns = chunk.select_dtypes(
        include=['object', 'string']
    ).columns

    # Normalize all text columns
    for col in text_columns:

        chunk[col] = chunk[col].astype(str).apply(normalize_text)

    # Save normalized chunk
    chunk.to_csv(
        output_file,
        mode='a',
        index=False,
        header=first_chunk
    )

    first_chunk = False

chunk.head()


print("Dataset normalization completed.")

Dataset normalization completed.


# Question 1

In this section, 10 random records are selected
from the dataset in order to analyze the structure
and informational layers of the Noushirvani Bank data.

In [4]:
# انتخاب ۱۰ رکورد تصادفی
for chunk in read_in_chunks("normalized_bank_data.csv" , 1000000) :

    sample_records = chunk.sample(10)

    break

# نمایش رکوردها

sample_records

,national_code,FULL_NAME,FATHER_NAME,BIRTH_DATE,CITY_NAME,PROVINCE_NAME,BIRTH_CITY,BIRTH_PROVINCE
774669,774670,امیر رضا,ناصر,1358-05-20,NaN,NaN,محلات,مرکزی
347540,347541,محمد شکری حسینی,خلیل,1363-04-17,NaN,NaN,ارومیه,آذربایجان غربی
436474,436475,هدایت گراوند,علی عباس,1357-09-05,تهران,تهران,کوهدشت,لرستان
142277,142278,یوسف کمالی فرد,حبیب اله,1370-08-26,NaN,NaN,ممسنی,فارس
197406,197407,زهرا زارع,علی همت,1374-03-08,شیراز,فارس,خرامه,فارس
40792,40793,مقصود علی پورنوتاش,علی,1337-02-02,تهران,تهران,تبریز,آذربایجان شرقی
823371,823372,ولی یوسفی,حمزه علی,1351-04-04,تهران,تهران,ترکمانچای,آذربایجان شرقی
678648,678649,مریم زهره ادبی,محمدرضا,1361-04-05,کرمان,کرمان,سیرجان,کرمان
275390,275391,فرزانه صادق زاده صفاریان,حمید,1362-06-23,NaN,NaN,مشهد,خراسان رضوی
917512,917513,فاطمه حسین پورنوا,فیض الله,1334-10-04,مراغه,اذربایجان شرقی,مراغه,آذربایجان شرقی


# Analizing the Records

In [ ]:
# Display dataset dimensions
for chunk in read_in_chunks("normalized_bank_data.csv" , 1000000) :
    print(chunk.shape)

    # Display column names

    print(chunk.columns)

    break

(1000000, 8)
Index(['national_code', 'FULL_NAME', 'FATHER_NAME', 'BIRTH_DATE', 'CITY_NAME',
       'PROVINCE_NAME', 'BIRTH_CITY', 'BIRTH_PROVINCE'],
      dtype='str')


# Question 2

In this section, missing residence information is handled.

For records where the residence city or province is missing,
the corresponding birth city and birth province values are used.

This helps maintain geographical consistency in the dataset.

In [ ]:
# Fill missing residence city using birth city
# only when birth city exists
sum1 = sum2 = 0
for chunk in read_in_chunks("normalized_bank_data.csv" , 1000000) :
    city_condition = (
        chunk['CITY_NAME'].isna() &
        chunk['BIRTH_CITY'].notna()
    )

    chunk.loc[city_condition, 'CITY_NAME'] = chunk.loc[
        city_condition,
        'BIRTH_CITY'
    ]


    # Fill missing residence province using birth province
    # only when birth province exists

    province_condition = (
        chunk['PROVINCE_NAME'].isna() &
        chunk['BIRTH_PROVINCE'].notna()
    )

    chunk.loc[province_condition, 'PROVINCE_NAME'] = chunk.loc[
        province_condition,
        'BIRTH_PROVINCE'
    ]


    # Display remaining missing values
    x , y = chunk[['CITY_NAME', 'PROVINCE_NAME']].isnull().sum()
    sum1 += x
    sum2 += y

print(f"CITY_NAME = {sum1} , PROVINCE_NAME = {sum2}")


print("--------------------")

CITY_NAME = 363515 , PROVINCE_NAME = 364645
--------------------


# Analysis

The missing residence locations were updated only for

records that had valid birth location information.

This method prevents invalid replacements and improves

the consistency of geographical data in the dataset.

# Question 3

## Description

The national code values in the dataset have different lengths.

Since the standard Iranian national code must contain
exactly 10 digits, all values in the national code column
are converted into a unified 10-digit format.

## Objective

Standardize all national codes in order to improve
data consistency and prevent identification errors.

In [8]:
for chunk in read_in_chunks("normalized_bank_data.csv" , 1000000) :
    # Convert national codes to string format

    chunk['national_code'] = chunk['national_code'].astype(str)

    # Remove extra spaces

    chunk['national_code'] = chunk['national_code'].str.strip()


    # Convert all national codes to 10 digits
    # by adding leading zeros

    chunk['national_code'] = chunk['national_code'].str.zfill(10)


    # Display sample national codes

chunk[['national_code']].head(10)

,national_code
32000000,0032000001
32000001,0032000002
32000002,0032000003
32000003,0032000004
32000004,0032000005
32000005,0032000006
32000006,0032000007
32000007,0032000008
32000008,0032000009
32000009,0032000010


# Question 4

## Description

In this section, the dataset is searched in order to
identify all individuals whose last name is "علیزاده"
and whose residence province is "هرمزگان".

## Objective

Filter suspicious records related to the financial
investigation described in the project scenario.

In [ ]:
# List for storing filtered records

alizadeh_people = []


for chunk in read_in_chunks("normalized_bank_data.csv", 1000000):

    # Extract last name
    chunk['LAST_NAME'] = chunk['FULL_NAME'].str.split().str[-1]

    # Filter records
    filtered = chunk[
        (chunk['LAST_NAME'].str.strip() == 'علیزاده') &
        (chunk['PROVINCE_NAME'].str.strip() == 'هرمزگان')
    ]

    # Store filtered chunk
    alizadeh_people.append(filtered)


# Merge all filtered chunks

final_fd = pd.concat(
    alizadeh_people,
    ignore_index=True
)


# Display result shape

print(final_fd.shape)


# Display sample results

final_fd.head()

(300, 9)


,national_code,FULL_NAME,FATHER_NAME,BIRTH_DATE,CITY_NAME,PROVINCE_NAME,BIRTH_CITY,BIRTH_PROVINCE,LAST_NAME
0,27291,سعید علیزاده,فریدون,1363-06-29,بندرعباس,هرمزگان,بندرعباس,هرمزگان,علیزاده
1,36467,امیرعباس علیزاده,علی,1385-10-14,بندرعباس,هرمزگان,بندرعباس,هرمزگان,علیزاده
2,115490,کبری حاجی علیزاده,مرید,1358-01-01,میناب,هرمزگان,شهربابک,کرمان,علیزاده
3,130116,معراج علیزاده,اسلام,1384-02-01,میناب,هرمزگان,میناب,هرمزگان,علیزاده
4,455097,مصطفی علیزاده,محمود,1367-06-30,بندرعباس,هرمزگان,بندرعباس,هرمزگان,علیزاده


# Question 5

## Description

In this section, individuals whose last name is
"حسنپور" and whose residence province is "اصفهان"
are identified from the dataset.

A new feature called "ADDRESS" is then added to
the dataset, and the value "خوابگاه امینیان" is
assigned to the matching records.

## Objective

Provide temporary residence information for the
identified individuals according to the project scenario.

In [ ]:
hasanpour_people = []

for chunk in read_in_chunks("normalized_bank_data.csv",1000000) :

    # Extract last name from full name

    chunk['LAST_NAME'] = chunk['FULL_NAME'].str.split().str[-1]


    # Create ADDRESS column if it does not exist

    chunk['ADDRESS'] = np.nan


    # Identify target individuals

    hasanpour_condition = (
        (chunk['LAST_NAME'] == 'حسنپور') &
        (chunk['PROVINCE_NAME'] == 'اصفهان')
    )



    chunk['ADDRESS'] = chunk['ADDRESS'].astype(str)


    # Assign address value

    chunk.loc[
        hasanpour_condition,
        'ADDRESS'
    ] = 'خوابگاه امینیان'


    # Display matching records

    hasanpour_people.append(chunk[hasanpour_condition])

fd = pd.concat(hasanpour_people)

print(fd.shape)

fd.head()

(9, 10)


,national_code,FULL_NAME,FATHER_NAME,BIRTH_DATE,CITY_NAME,PROVINCE_NAME,BIRTH_CITY,BIRTH_PROVINCE,LAST_NAME,ADDRESS
1670939,1670940,قاسم حسنپور,خداکرم,1355-03-01,خمینی شهر,اصفهان,اصفهان,اصفهان,حسنپور,خوابگاه امینیان
9095145,9095146,محمد حسنپور,امراله,1369-07-05,فلاورجان,اصفهان,فلاورجان,اصفهان,حسنپور,خوابگاه امینیان
11996699,11996700,محمد حسنپور,امراله,1369-07-05,فلاورجان,اصفهان,فلاورجان,اصفهان,حسنپور,خوابگاه امینیان
18220278,18220279,محمد حسنپور,امراله,1369-07-05,فلاورجان,اصفهان,فلاورجان,اصفهان,حسنپور,خوابگاه امینیان
19110094,19110095,پردیس حسنپور,محمد,1366-03-25,اصفهان,اصفهان,فلاورجان,اصفهان,حسنپور,خوابگاه امینیان


applying the changes into the main NIT Bank file code

In [ ]:
import os

temp_file = "temp.csv"

first_chunk = True

for chunk in read_in_chunks(
    "Bank NIT DB.csv",
    1000000
):

    # شرط حسنپور
    condition = (
        chunk['FULL_NAME']
        .astype(str)
        .str.contains('حسنپور', na=False)
    )

    # ایجاد ستون ADDRESS
    chunk.loc[
        condition,
        'ADDRESS'
    ] = 'خوابگاه امینیان'

    # ذخیره chunk
    chunk.to_csv(
        temp_file,
        mode='a',
        index=False,
        header=first_chunk
    )

    first_chunk = False


# جایگزینی فایل اصلی

os.remove("Bank NIT DB.csv")

os.rename(temp_file, "Bank NIT DB.csv")

# Question 6

## Description

In this section, a new column named "CARD_NUMBER"
is added to the dataset.

All generated card numbers must:

- Start with the prefix "3550"
- Be unique for every individual
- Follow a standard 16-digit bank card format

Because the dataset is extremely large, the operation
is performed using chunk-based processing.

In [ ]:
# Library for random number generation

import random


# Output file

output_file = "bank_data_with_cards.csv"


# To write header only once

first_chunk = True


# Set for storing unique card numbers

generated_cards = set()


# Function for generating unique card numbers

def generate_card_number():

    while True:

        # Generate remaining 12 digits
        random_digits = ''.join(

            random.choices('0123456789', k=12)

        )

        # Create full card number
        card_number = '3550' + random_digits

        # Check uniqueness
        if card_number not in generated_cards:

            generated_cards.add(card_number)

            return card_number


# Process dataset in chunks

for chunk in read_in_chunks(
    "processed_bank_data.csv",
    1000000
):

    # Generate unique card numbers
    chunk['CARD_NUMBER'] = [

        generate_card_number()

        for _ in range(len(chunk))
    ]

    # Save processed chunk
    chunk.to_csv(
        output_file,
        mode='a',
        index=False,
        header=first_chunk
    )

    first_chunk = False


print("Question 6 processing completed.")